In [ ]:
# Setup: code comes from GitHub onto the VM disk; data + checkpoints live on Drive (see src/config.py)
import os, sys
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = "/content/NeoAntigen"
try:   # private repo: store a GitHub token as Colab secret GH_TOKEN (never write it in the notebook)
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = ""
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/Tienanh209/NeoAntigen.git"
if not os.path.exists(REPO_DIR):
    !git clone {URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)
!apt-get -qq install -y cd-hit > /dev/null

In [ ]:
# Download once (dated snapshot on Drive): lung-cancer neoantigens (positives) + viral/TAA epitopes (negatives)
import pandas as pd
from src import config, lung
pd.set_option("display.width", 200)

raw = lung.get_raw(config.DATA_DIR, refresh=False)
print(raw.groupby("group").size())
# Optional extra positives, downloaded by hand into DATA_DIR/lung_extra/ (TSNAdb is *predicted*, not validated):
# extra = lung.load_extra(config.DATA_DIR / "lung_extra" / "tsnadb_lung.tsv", "tsnadb", seq_col="peptide", hla_col="HLA")
# raw = lung.add_extra(raw, [extra])     # then use LungPreset(..., extra_sources=("tsnadb",))
raw = lung.add_extra(raw, [])
lung.sweep(raw)

In [ ]:
# Run the full pipeline for one preset; output goes to DATA_DIR/<name>/ (data.load_dataset(config.DATA_DIR / PRESET))
PRESET = "lung_cedar"
df, final, tables = lung.run(raw, lung.PRESETS[PRESET], config.DATA_DIR, config.WORK_DIR)
for t in tables:
    print(t.to_string(index=False), "\n")